# Environment preparation

This administrative notebook prepares the environment-specific Unity Catalog boundary for the SalesLT, Sales JSON, and Sales CSV pipelines. It resolves DEV or PROD resources, creates governed ADLS external locations, validates connectivity, and creates the catalogs and Medallion schemas that runtime jobs use later.

In [0]:
# Environment bootstrap (administrative phase).
# Run this notebook with a Unity Catalog administrator or delegated deployment identity, not the ETL runtime principal.
# The DEV/PROD mapping prevents resources from being created against the wrong workspace or storage account.
# Before execution, the workspace must use the intended metastore and the ADLS containers must already exist.
# Each referenced Storage Credential is expected to use the environment-specific Azure Managed Identity.

dbutils.widgets.text("environment", "dev", "Environment")

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError("Environment must be either 'dev' or 'prod'.")

config = {
    "dev": {
    "workspace": "dbw-centralus-dev01",
    "storage_account": "stcentralusjrdev",
    "storage_credential": "dbac_centralus_dbx_dev",

    "external_locations": {
        "landing": "ext_landing_dev",
        "lakehouse": "ext_lakehouse_dev",
        "streaming": "ext_streaming_dev"
    },

    "catalogs": {
        "saleslt": "saleslt_dev",
        "salesjson": "salesjson_dev",
        "salescsv": "salescsv_dev"
    }
},
    "prod": {
        "workspace": "dbw-centralus-prod01",
        "storage_account": "stcentralusjrprod",
        "storage_credential": "dbac_centralus_dbx_pro",
        "external_locations": {
            "landing": "ext_landing_prod",
            "lakehouse": "ext_lakehouse_prod",
            "streaming": "ext_streaming_prod"
        }, 
        "catalogs": {
    "saleslt": "saleslt_prod",
    "salesjson": "salesjson_prod",
    "salescsv": "salescsv_prod"
}
    }
}

env = config[environment]

print(f"Environment: {environment}")
print(f"Expected workspace: {env['workspace']}")
print(f"Storage account: {env['storage_account']}")
print(f"Storage credential: {env['storage_credential']}")

In [0]:
# Bind governed Unity Catalog names to ADLS paths through the managed-identity-backed Storage Credential.
# Landing holds source files, lakehouse holds Delta data, and streaming holds checkpoints and schema metadata.
# CREATE IF NOT EXISTS keeps this administrative setup safe to rerun.

storage_account = env["storage_account"]
storage_credential = env["storage_credential"]

external_locations = {
    env["external_locations"]["landing"]: {
        "url": f"abfss://landing@{storage_account}.dfs.core.windows.net/",
        "comment": (
            f"{environment.upper()} landing zone for incoming source files. "
            "Used as the raw file ingestion area for JSON and CSV datasets."
        )
    },

    env["external_locations"]["lakehouse"]: {
        "url": f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/",
        "comment": (
            f"{environment.upper()} external Delta Lake storage location for "
            "Bronze, Silver, and Gold tables governed by Unity Catalog."
        )
    },

    env["external_locations"]["streaming"]: {
        "url": f"abfss://streaming@{storage_account}.dfs.core.windows.net/",
        "comment": (
            f"{environment.upper()} streaming metadata location used for "
            "Auto Loader checkpoints and schema tracking."
        )
    }
}

for location_name, properties in external_locations.items():

    statement = f"""
    CREATE EXTERNAL LOCATION IF NOT EXISTS `{location_name}`
    URL '{properties["url"]}'
    WITH (STORAGE CREDENTIAL `{storage_credential}`)
    COMMENT '{properties["comment"]}'
    """

    print(f"Creating external location: {location_name}")

    spark.sql(statement)

In [0]:
# Validate storage access under the current administrator/deployment identity before creating governed objects.
# Successful listings confirm that the workspace, Storage Credential, Managed Identity, and ADLS permissions align.
test_paths = [
    f"abfss://landing@{storage_account}.dfs.core.windows.net/",
    f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/",
    f"abfss://streaming@{storage_account}.dfs.core.windows.net/"
]

for path in test_paths:

    print(f"\nTesting: {path}")

    try:
        files = dbutils.fs.ls(path)
        print(f"SUCCESS - Accessible: {path}")

        for item in files[:10]:
            print(item.path)

    except Exception as e:
        print(f"FAILED - {path}")
        print(str(e))

In [0]:
# Create one catalog per pipeline and environment, each with an isolated managed root in the lakehouse container.
# Managed locations keep catalog-owned tables governed by Unity Catalog while separating DEV and PROD data.

storage_account = env["storage_account"]

catalog_definitions = {
    env["catalogs"]["saleslt"]: {
        "managed_location": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            f"_managed/{env['catalogs']['saleslt']}/"
        ),
        "comment": (
            f"{environment.upper()} catalog for the SalesLT batch ETL pipeline. "
            "Stores Bronze, Silver, and Gold Delta tables derived from the "
            "Azure SQL SalesLT source."
        )
    },

    env["catalogs"]["salesjson"]: {
        "managed_location": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            f"_managed/{env['catalogs']['salesjson']}/"
        ),
        "comment": (
            f"{environment.upper()} catalog for the JSON sales streaming pipeline. "
            "Stores Bronze, Silver, and Gold Delta tables populated from JSON "
            "sales events ingested from Azure Data Lake Storage."
        )
    },

    env["catalogs"]["salescsv"]: {
        "managed_location": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            f"_managed/{env['catalogs']['salescsv']}/"
        ),
        "comment": (
            f"{environment.upper()} catalog for the CSV batch ingestion pipeline. "
            "Stores Bronze, Silver, and Gold Delta tables populated from CSV "
            "business datasets stored in Azure Data Lake Storage."
        )
    }
}


for catalog_name, properties in catalog_definitions.items():

    statement = f"""
    CREATE CATALOG IF NOT EXISTS `{catalog_name}`
    MANAGED LOCATION '{properties["managed_location"]}'
    COMMENT '{properties["comment"]}'
    """

    print(f"Creating catalog: {catalog_name}")

    spark.sql(statement)

In [0]:
# Create the Bronze, Silver, and Gold schemas used by every pipeline within each environment-specific catalog.
# Permissions are intentionally applied later by the security notebook so administration stays auditable and
# runtime identities receive only the privileges required to execute ETL workloads.
schema_definitions = {

    "bronze": (
        "Bronze layer containing raw or minimally processed data ingested from "
        "the source system. Data is preserved as close as possible to its "
        "original structure for traceability and reprocessing."
    ),

    "silver": (
        "Silver layer containing validated, cleaned, standardized, deduplicated, "
        "and enriched data prepared for downstream analytical processing."
    ),

    "gold": (
        "Gold layer containing curated business-level datasets, aggregations, "
        "and analytical models optimized for Databricks Genie, dashboards, "
        "Power BI, and downstream consumers."
    )
}

for catalog_name in catalog_definitions.keys():

    for schema_name, schema_comment in schema_definitions.items():

        statement = f"""
        CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema_name}`
        COMMENT '{schema_comment}'
        """

        print(f"Creating schema: {catalog_name}.{schema_name}")

        spark.sql(statement)
